# Text Preprocessing

This notebook cleans and normalizes the page-level text extracted from the Porto Municipal Regulatory Code. It removes recurring page artifacts while preserving the legal structure required for article parsing.

The output is `data/processed/crmp_cleaned.json`, which retains page boundaries and document metadata for the next pipeline stage.


## 1. Import the required libraries

Load path, JSON, and regular-expression utilities used throughout preprocessing.


In [1]:
from pathlib import Path
import json
import re


## 2. Define the data paths

Resolve the extracted input file and the destination for the cleaned page dataset.


In [2]:
# Run this notebook from notebooks/ so the project root is its parent.
PROJECT_ROOT = Path.cwd().parent

INPUT_FILE = PROJECT_ROOT / "data" / "processed" / "crmp_extracted.json"
OUTPUT_FILE = PROJECT_ROOT / "data" / "processed" / "crmp_cleaned.json"

print(INPUT_FILE)
print(OUTPUT_FILE)


c:\Users\user\Documents\GitHub\legal-rag-pt\data\processed\crmp_extracted.json
c:\Users\user\Documents\GitHub\legal-rag-pt\data\processed\crmp_cleaned.json


## 3. Load the extracted pages

Read the page-level JSON produced by the extraction notebook and select its page records.


In [3]:
with open(INPUT_FILE, "r", encoding="utf-8") as f:
    data = json.load(f)

pages = data["pages"]

print(f"Páginas carregadas: {len(pages)}")


Páginas carregadas: 662


## 4. Inspect representative source pages

Review pages with different layouts to identify recurring headers and formatting artifacts.


In [4]:
for page_number in [22, 23, 24, 26]:
    # Stored pages use one-based labels, while Python lists are zero-based.
    page = pages[page_number - 1]

    print("=" * 100)
    print(f"PAGE {page_number}")
    print("=" * 100)
    print(page["text"][:2500])
    print()


PAGE 22
Código Regulamentar do Município do Porto | Parte A | A.1. Princípios gerais  
                       22 
 
 
Parte A 
Parte Geral  
Código Regulamentar do Município do Porto 
 
PARTE A 
Parte geral 
 
Artigo A/1.º 
Objeto do código 
1 – O presente código consagra as disposições regulamentares com eficácia externa em 
vigor na área do Município do Porto nos seguintes domínios: 
a) Urbanismo;                                                          
b) Ambiente; 
c) Gestão do espaço público; 
d) Intervenção municipal sobre o exercício de atividades privadas; 
e) Gestão de recursos; 
f) Taxas e outras receitas municipais; 
g) Fiscalização e sancionamento de infrações. 
2 – Esta codificação não prejudica a existência, nos domínios referidos, de disposições 
regulamentares complementares ao presente código, nele devidamente referenciadas.  
 
Artigo A/2.º 
Objeto da Parte A 
A Parte A consagra:  
a) No Título I, os princípios gerais inspiradores do código, que, para além dos 
princ

## 5. Define general text normalization

Create a reusable function for whitespace cleanup and removal of common invisible characters.


In [5]:
def clean_text(text):
    # Remove espaços no início/fim
    text = text.strip()

    # Remove caracteres invisíveis comuns
    # Remove soft hyphens introduced by PDF text extraction.
    text = text.replace("\u00ad", "")
    text = text.replace("\ufeff", "")
    text = text.replace("\xa0", " ")

    # Normaliza espaços horizontais
    text = re.sub(r"[ \t]+", " ", text)

    # Collapse repeated blank lines.
    text = re.sub(r"\n{3,}", "\n\n", text)

    # Remove espaços antes de newline
    text = re.sub(r" +\n", "\n", text)

    return text.strip()


## 6. Remove recurring page headers

Filter lines that match known CRMP header patterns without altering substantive legal text.


In [6]:
def remove_page_headers(text):
    lines = text.splitlines()

    cleaned_lines = []

    for line in lines:
        stripped = line.strip()

        # Cabeçalho com separadores |
        if (
            stripped.startswith("Código Regulamentar do Município do Porto")
            and "|" in stripped
        ):
            continue

        # Cabeçalho simples isolado
        if stripped == "Código Regulamentar do Município do Porto":
            continue

        cleaned_lines.append(line)

    # Rebuild the page while preserving meaningful line boundaries.
    return "\n".join(cleaned_lines)


## 7. Remove isolated page numbers

Discard lines that contain only a printed page number and would otherwise pollute the corpus.


In [7]:
def remove_isolated_page_numbers(text):
    lines = text.splitlines()

    cleaned_lines = []

    for line in lines:
        stripped = line.strip()

        if re.fullmatch(r"\d{1,3}", stripped):
            continue

        cleaned_lines.append(line)

    return "\n".join(cleaned_lines)


## 8. Combine the preprocessing steps

Apply header removal, page-number removal, and general normalization in a consistent order.


In [8]:
def preprocess_page(text):
    text = remove_page_headers(text)
    text = remove_isolated_page_numbers(text)
    text = clean_text(text)

    return text


## 9. Clean every page

Process the complete document while preserving each original page number.


In [9]:
cleaned_pages = []

for page in pages:
    cleaned_text = preprocess_page(page["text"])

    cleaned_pages.append({
        "page": page["page"],
        "text": cleaned_text
    })

print(f"Páginas processadas: {len(cleaned_pages)}")


Páginas processadas: 662


## 10. Compare original and cleaned text

Inspect a representative page before and after preprocessing to validate the transformation.


In [10]:
page_number = 22

original = pages[page_number - 1]["text"]
cleaned = cleaned_pages[page_number - 1]["text"]

print("ORIGINAL")
print("=" * 100)
print(original[:3000])

print("\n\nCLEANED")
print("=" * 100)
print(cleaned[:3000])


ORIGINAL
Código Regulamentar do Município do Porto | Parte A | A.1. Princípios gerais  
                       22 
 
 
Parte A 
Parte Geral  
Código Regulamentar do Município do Porto 
 
PARTE A 
Parte geral 
 
Artigo A/1.º 
Objeto do código 
1 – O presente código consagra as disposições regulamentares com eficácia externa em 
vigor na área do Município do Porto nos seguintes domínios: 
a) Urbanismo;                                                          
b) Ambiente; 
c) Gestão do espaço público; 
d) Intervenção municipal sobre o exercício de atividades privadas; 
e) Gestão de recursos; 
f) Taxas e outras receitas municipais; 
g) Fiscalização e sancionamento de infrações. 
2 – Esta codificação não prejudica a existência, nos domínios referidos, de disposições 
regulamentares complementares ao presente código, nele devidamente referenciadas.  
 
Artigo A/2.º 
Objeto da Parte A 
A Parte A consagra:  
a) No Título I, os princípios gerais inspiradores do código, que, para além dos 
prin

## 11. Define the article-heading pattern

Compile a regular expression for the article identifiers used in the CRMP.


In [11]:
ARTICLE_PATTERN = re.compile(
    r"^Artigo\s+([A-Z](?:-\d+)?/\d+\.º(?:-[A-Z])?)$",
    re.IGNORECASE
)


## 12. Validate article detection

Scan the cleaned pages and display detected article headings as a preprocessing quality check.


In [12]:
for page in cleaned_pages:
    for line in page["text"].splitlines():
        match = ARTICLE_PATTERN.match(line.strip())

        if match:
            print(page["page"], "->", line.strip())


22 -> Artigo A/1.º
22 -> Artigo A/2.º
22 -> Artigo A-1/1.º
23 -> Artigo A-1/2.º
23 -> Artigo A-1/3.º
23 -> Artigo A-1/4.º
23 -> Artigo A-1/5.º
24 -> Artigo A-1/6.º
24 -> Artigo A-1/7.º
26 -> Artigo A-2/1.º
27 -> Artigo A-2/2.º
27 -> Artigo A-2/3.º
27 -> Artigo A-2/4.º
28 -> Artigo A-2/5.º
29 -> Artigo A-2/6.º
29 -> Artigo A-2/7.º
29 -> Artigo A-2/8.º
30 -> Artigo A-2/9.º
30 -> Artigo A-2/10.º
30 -> Artigo A-2/11.º
31 -> Artigo A-2/12.º
32 -> Artigo A-2/13.º
32 -> Artigo A-2/13.º-A
33 -> Artigo A-2/14.º
33 -> Artigo A-2/15.º
33 -> Artigo A-2/15.º-A
34 -> Artigo A-2/16.º
34 -> Artigo A-2/17.º
35 -> Artigo B-1/1.º
35 -> Artigo B-1/2.º
36 -> Artigo B-1/2.º-A
37 -> Artigo B-1/3.º
38 -> Artigo B-1/4.º
38 -> Artigo B-1/5.º
39 -> Artigo B-1/5.º-A
40 -> Artigo B-1/6.º
40 -> Artigo B-1/7.º
40 -> Artigo B-1/8.º
42 -> Artigo B-1/9.º
43 -> Artigo B-1/10.º
43 -> Artigo B-1/11.º
43 -> Artigo B-1/12.º
43 -> Artigo B-1/13.º
43 -> Artigo B-1/14.º
44 -> Artigo B-1/15.º
45 -> Artigo B-1/16.º
45 -> Artigo 

## 13. Define the part-heading pattern

Compile a regular expression for top-level legal divisions such as `PARTE A`.


In [13]:
PART_PATTERN = re.compile(
    r"^PARTE\s+([A-Z])$",
    re.IGNORECASE
)

TITLE_PATTERN = re.compile(
    r"^TÍTULO\s+([IVXLCDM]+)$",
    re.IGNORECASE
)


## 14. Validate part detection

Scan the cleaned text and report the pages on which part headings are found.


In [14]:
for page in cleaned_pages:
    for line in page["text"].splitlines():
        stripped = line.strip()

        if PART_PATTERN.match(stripped):
            print("PART:", page["page"], stripped)

        if TITLE_PATTERN.match(stripped):
            print("TITLE:", page["page"], stripped)


PART: 22 Parte A
PART: 22 PARTE A
TITLE: 22 TÍTULO I
PART: 23 Parte A
PART: 24 Parte A
PART: 25 Parte A
PART: 26 Parte A
TITLE: 26 TÍTULO II
PART: 27 Parte A
PART: 28 Parte A
PART: 29 Parte A
PART: 30 Parte A
PART: 31 Parte A
PART: 32 Parte A
PART: 33 Parte A
PART: 34 Parte A
PART: 35 Parte B
PART: 35 PARTE B
TITLE: 35 TÍTULO I
PART: 36 Parte B
PART: 37 Parte B
PART: 38 Parte B
PART: 39 Parte B
PART: 40 Parte B
PART: 41 Parte B
PART: 42 Parte B
PART: 43 Parte B
PART: 44 Parte B
PART: 45 Parte B
PART: 46 Parte B
PART: 47 Parte B
PART: 48 Parte B
PART: 49 Parte B
PART: 50 Parte B
PART: 51 Parte B
PART: 52 Parte B
PART: 53 Parte B
PART: 54 Parte B
PART: 55 Parte B
PART: 56 Parte B
PART: 57 Parte B
PART: 58 Parte B
PART: 59 Parte B
TITLE: 59 TÍTULO II
PART: 60 Parte B
PART: 61 Parte B
PART: 62 Parte B
PART: 63 Parte B
PART: 64 Parte B
PART: 65 Parte B
PART: 66 Parte C
PART: 66 PARTE C
TITLE: 66 TÍTULO I
PART: 67 Parte C
PART: 68 Parte C
PART: 69 Parte C
PART: 70 Parte C
PART: 71 Parte C
TI

## 15. Build the cleaned document object

Combine the original metadata with the normalized page records.


In [15]:
output = {
    "document": data["document"],
    "source_file": data["source_file"],
    "num_pages": len(cleaned_pages),
    "pages": cleaned_pages
}


## 16. Save the cleaned dataset

Serialize the processed pages as readable UTF-8 JSON for document parsing.


In [16]:
OUTPUT_FILE.parent.mkdir(parents=True, exist_ok=True)

with open(OUTPUT_FILE, "w", encoding="utf-8") as f:
    json.dump(
        output,
        f,
        ensure_ascii=False,  # Keep Portuguese characters readable in the JSON file.
        indent=2
    )

print(f"Ficheiro criado: {OUTPUT_FILE}")


Ficheiro criado: c:\Users\user\Documents\GitHub\legal-rag-pt\data\processed\crmp_cleaned.json
